# PBC cohort: modeling analysis

This notebook uses the maintained production modules in `src/` for loading,
splitting, preprocessing, fitting, threshold selection, and metrics. It is
runnable from either `cirrhosis/` or `cirrhosis/notebooks/`.

The primary endpoint is **Stage 3–4 versus Stage 1–2**. Exact four-stage and
cumulative ordinal models are secondary. This is exploratory research with no
external or temporal validation: **not for clinical use, not medical advice,
and not a replacement for biopsy or clinician assessment**. See
[AASLD PBC guidance](https://www.aasld.org/practice-guidelines/primary-biliary-cholangitis)
and [EASL PBC guidance](https://easl.eu/publication/management-of-cholestatic-liver-diseases/).

**Notebook contract:** every executable cell is immediately followed by a short
discussion of what the output shows, what it means clinically, and what was
decided as a result.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents]
                    if (p / "src" / "data.py").exists() and (p / "data" / "raw" / "pbc.csv").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import events_per_variable, load_pbc_data, train_test_split_pipeline
from src.evaluation import (
    binary_metrics,
    bootstrap_metric_intervals,
    calibration_metrics,
    multiclass_metrics,
    select_operating_threshold,
)
from src.modeling import fit_model

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "pbc.csv"
frame = load_pbc_data(DATA_PATH)
MODELS = ("logistic", "random_forest", "hist_gradient_boosting")
print(f"Project root: {PROJECT_ROOT}; validated rows={len(frame)}")
print(f"Models compared: {MODELS}")


Project root: /home/datakrdo/Documents/portfolio/cirrhosis; validated rows=418
Models compared: ('logistic', 'random_forest', 'hist_gradient_boosting')


The notebook resolves its root from the presence of `src/data.py` and the
validated CSV, so execution location cannot silently select a different
dataset.

Reproducibility matters because model claims from a small historical cohort
are especially sensitive to data provenance.

Only the production loaders and estimators are used; core preprocessing or
model logic is never copied into notebook cells.

In [2]:
split = train_test_split_pipeline(frame, test_size=0.2, random_state=41)
epv = events_per_variable(split.X_train, split.y_binary_train)
print({
    "train_rows": len(split.X_train), "test_rows": len(split.X_test),
    "excluded_predictor_columns": ["Stage", "ID", "N_Days", "Status", "Drug"],
    "added_predictor_columns": ["trial_cohort"],
    "train_test_index_overlap": len(set(split.X_train.index) & set(split.X_test.index)),
    "events_per_variable_train": round(epv, 3),
})
print("Train endpoint counts:", split.y_binary_train.value_counts().sort_index().to_dict())
print("Test endpoint counts:", split.y_binary_test.value_counts().sort_index().to_dict())
print("Train cohort counts:", split.X_train["trial_cohort"].value_counts().to_dict())
print("Test cohort counts:", split.X_test["trial_cohort"].value_counts().to_dict())


{'train_rows': 329, 'test_rows': 83, 'excluded_predictor_columns': ['Stage', 'ID', 'N_Days', 'Status', 'Drug'], 'added_predictor_columns': ['trial_cohort'], 'train_test_index_overlap': 0, 'events_per_variable_train': 5.625}
Train endpoint counts: {0: 90, 1: 239}
Test endpoint counts: {0: 23, 1: 60}
Train cohort counts: {'randomised': 249, 'registry': 80}
Test cohort counts: {'randomised': 63, 'registry': 20}


Labeled patients are split once, stratified on endpoint x `trial_cohort` so
both the randomised and registry subcohorts are represented in train and test;
`Drug` is excluded as a leakage column (it proxies cohort membership almost
perfectly, see `01_eda.ipynb`) and `trial_cohort` is included in its place.
Events-per-variable on the training fold is below the conventional EPV >= 10
rule of thumb, which is reported rather than hidden.

Follow-up time and status can encode disease outcome after baseline and would
inflate apparent performance; test labels must remain untouched until final
reporting.

All imputers/encoders/models are fit on training rows only, the operating
threshold is selected by cross-validation on the training fold only (no manual
validation split), and the test set is evaluated once.

In [3]:
from src.modeling import build_named_pipeline

thresholds = {}
final_models = {}
test_probabilities = {}
for name in MODELS:
    pipeline = build_named_pipeline(name, split.X_train, random_state=41)
    threshold, tuned = select_operating_threshold(
        pipeline, split.X_train, split.y_binary_train.to_numpy(), random_state=41
    )
    thresholds[name] = threshold
    final_models[name] = tuned
    test_probabilities[name] = tuned.predict_proba(split.X_test)[:, 1]
print("CV-tuned decision thresholds (TunedThresholdClassifierCV, balanced_accuracy):")
print({k: round(v, 4) for k, v in thresholds.items()})


CV-tuned decision thresholds (TunedThresholdClassifierCV, balanced_accuracy):
{'logistic': 0.5472, 'random_forest': 0.538, 'hist_gradient_boosting': 0.5329}


All three models are trained through `build_named_pipeline` (fresh
fold-fitted preprocessing per model — median-impute/one-hot for
`logistic`/`random_forest`, no imputation and native categorical dtype for
`hist_gradient_boosting`) and their operating threshold is chosen by
`TunedThresholdClassifierCV`, which cross-validates on the training fold only
— no manual validation split, no test label ever touches threshold selection.

A transparent linear baseline, a nonlinear tree ensemble, and a
native-missingness gradient booster test whether relationships beyond additive
effects — and beyond what median imputation preserves — matter, without
implying causality or individual diagnosis.

All three prespecified models carry into held-out comparison; `hist_gradient_boosting`
is the primary comparator (see `README.md`), the others are baselines. No model
is selected based on test performance.

In [4]:
held_out = {}
for name, probabilities in test_probabilities.items():
    metrics = binary_metrics(split.y_binary_test.to_numpy(), probabilities, threshold=thresholds[name])
    metrics["bootstrap_95_ci"] = bootstrap_metric_intervals(
        split.y_binary_test.to_numpy(), probabilities, threshold=thresholds[name],
        n_bootstrap=100, random_state=41
    )
    metrics["calibration"] = calibration_metrics(split.y_binary_test.to_numpy(), probabilities)
    held_out[name] = metrics
print(json.dumps(held_out, indent=2))
artifact = PROJECT_ROOT / "outputs" / "notebook_modeling_metrics.json"
artifact.write_text(json.dumps({"primary_endpoint": "Stage 3-4 vs Stage 1-2", "models": held_out}, indent=2) + "\n")
print(f"Wrote {artifact}")


{
  "logistic": {
    "auroc": 0.777536231884058,
    "auprc": 0.9106872897288659,
    "sensitivity": 0.5833333333333334,
    "specificity": 0.7391304347826086,
    "threshold": 0.5472444899273095,
    "positive_rate": 0.4939759036144578,
    "brier_score": 0.19513379136813166,
    "bootstrap_95_ci": {
      "auroc": [
        0.6728385416666667,
        0.900969696969697
      ],
      "auprc": [
        0.8453864726812629,
        0.9611308762961732
      ],
      "sensitivity": [
        0.4576271186440678,
        0.7184029807130331
      ],
      "specificity": [
        0.575,
        0.9047727272727272
      ]
    },
    "calibration": {
      "brier_score": 0.19513379136813166,
      "fraction_of_positives": [
        0.5555555555555556,
        0.25,
        0.5,
        0.75,
        0.8888888888888888,
        0.625,
        0.625,
        1.0,
        1.0,
        1.0
      ],
      "mean_predicted_value": [
        0.2579688127044452,
        0.33730721688734194,
        0

AUROC, AUPRC, sensitivity, specificity, and percentile bootstrap intervals
are computed on the untouched test predictions; intervals may be wide because
only about one fifth of 412 labeled rows are held out.

Uncertainty and class imbalance make a single point estimate unsafe for
clinical decisions, especially in a treatment-era cohort without external
validation.

The complete held-out metric set is reported with intervals, without declaring
superiority from small differences, and none of it is used or deployed for
patient care.

In [5]:
secondary = {}
multiclass_model = fit_model("multiclass_logistic", split.X_train, split.y_stage_train, random_state=41)
secondary["multiclass_logistic"] = multiclass_metrics(
    split.y_stage_test.to_numpy(), np.asarray(multiclass_model.predict(split.X_test))
)
ordinal_model = fit_model("ordinal_logistic", split.X_train, split.y_stage_train, random_state=41)
secondary["ordinal_logistic"] = multiclass_metrics(
    split.y_stage_test.to_numpy(), np.asarray(ordinal_model.predict(split.X_test))
)
print(json.dumps(secondary, indent=2))
artifact = PROJECT_ROOT / "outputs" / "notebook_modeling_metrics.json"
payload = json.loads(artifact.read_text())
payload["secondary"] = secondary
artifact.write_text(json.dumps(payload, indent=2) + "\n")


{
  "multiclass_logistic": {
    "macro_f1": 0.242966042966043,
    "balanced_accuracy": 0.2549401295557155,
    "quadratic_weighted_kappa": 0.34545454545454546
  },
  "ordinal_logistic": {
    "macro_f1": 0.257328278322926,
    "balanced_accuracy": 0.27106916181378005,
    "quadratic_weighted_kappa": 0.35185185185185186
  }
}


4799

Exact four-stage and cumulative ordinal estimators use the same
training/test split and production preprocessing, and are summarized with
macro F1, balanced accuracy, and quadratic-weighted kappa rather than the
binary threshold.

Stage ordering carries clinical meaning, but adjacent-stage ambiguity and
sparse early classes limit certainty; ordinal agreement is not proof of valid
staging.

These analyses stay secondary and hypothesis-generating; the binary endpoint
remains the sole primary claim.

In [6]:
preprocessor = final_models["logistic"].estimator_.named_steps["preprocess"]
feature_names = list(preprocessor.get_feature_names_out())
coefficients = final_models["logistic"].estimator_.named_steps["model"].coef_[0]
importance = pd.Series(coefficients, index=feature_names).sort_values(key=np.abs, ascending=False).head(12)
print("Largest absolute logistic coefficients (direction is model association, not causation):")
print(importance.round(3).to_string())
print("\nNote: hist_gradient_boosting (the primary comparator) exposes neither coef_ nor "
      "feature_importances_; its explanations come from SHAP in 03_model_development.ipynb.")
limitations = [
    "418 source rows, 412 labeled Stage values, and substantial -- and structural, not random -- missingness.",
    "Historical 1974-1984 Mayo cohort; spectrum and treatment-era transportability are unknown.",
    "Events-per-variable on the training fold is below the EPV >= 10 rule of thumb.",
    "Single random holdout with no temporal or external validation; nested-CV and bootstrap-optimism "
    "estimates in 03_model_development.ipynb are the more defensible headline numbers.",
    "Associations may reflect confounding, measurement availability, and trial-cohort membership.",
    "No biopsy replacement, diagnosis, treatment recommendation, or clinical deployment.",
]
print("\nLimitations:")
for item in limitations:
    print(f"- {item}")


Largest absolute logistic coefficients (direction is model association, not causation):
Hepatomegaly_Y    0.387
Platelets        -0.347
Hepatomegaly_N   -0.318
Copper            0.245
Spiders_Y         0.222
Edema             0.208
Age               0.172
Albumin          -0.155
Spiders_N        -0.153
Ascites_N         0.139
Tryglicerides     0.117
Sex_M            -0.113

Note: hist_gradient_boosting (the primary comparator) exposes neither coef_ nor feature_importances_; its explanations come from SHAP in 03_model_development.ipynb.

Limitations:
- 418 source rows, 412 labeled Stage values, and substantial -- and structural, not random -- missingness.
- Historical 1974-1984 Mayo cohort; spectrum and treatment-era transportability are unknown.
- Events-per-variable on the training fold is below the EPV >= 10 rule of thumb.
- Single random holdout with no temporal or external validation; nested-CV and bootstrap-optimism estimates in 03_model_development.ipynb are the more defensible h

Coefficient rankings summarize one fitted model after one split and are
sensitive to correlated laboratory variables, missingness, and sample size;
they are not a stable feature-importance claim.

PBC staging requires clinical context and validated diagnostic pathways;
AASLD/EASL guidance, clinician judgment, and appropriate investigations
supersede this exploratory model.

These coefficients are used only to generate research hypotheses. External and
temporal validation plus calibration are required before any translational
discussion, and this analysis is not a biopsy replacement or for clinical use.